In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["INTECOMM_ENV"]
analysis_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
reports_folder = Path(os.environ["INTECOMM_ANALYSIS_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option('future.no_silent_downcasting', True)

**Notes:**

* Using retained_12m and retained_6m variables from df_main_1858
* Numbers correspond with the primary paper consort chart
* See get_df_main_1858() method retention where ``retained_12m`` = ``retained_months`` >= 11 and ``endline_visit_code`` >= 1120.0
* the 6m mark does not filter in the visit code since participants in the facility-care arm were on a routine schedule determined prior to the study start and did not always include study defined 1060 visit.



In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858

In [ ]:
df_main = get_df_main_1858(None, fasting_hours=8.0)

In [ ]:
df_main.query("primary_cohort.isin([1,2,3,4])").retained_12m.value_counts()

In [ ]:
df_main.query("primary_cohort.isin([1,2,3,4])").retained_6m.value_counts()


In [ ]:
df_main.query("primary_cohort.isin([1,2,3,4])").groupby(["primary_cohort_str", "assignment"]).size()

In [ ]:
df_main.query("primary_cohort.isin([1,2,3,4]) and retained_12m==1").groupby(["primary_cohort_str", "assignment"]).size()

In [ ]:
def get_onstudy_at(attr:str, cohorts:list[str], arm:str)->pd.DataFrame:
    if attr == "6m":
        df = df_main.query("primary_cohort_str.isin(@cohorts) and retained_6m==1 and assignment==@arm")
    elif attr == "12m":
        df = df_main.query("primary_cohort_str.isin(@cohorts) and retained_12m==1 and assignment==@arm")
    return df

def get_offstudy_reasons(attr:str, cohorts:list[str], arm:str)->pd.DataFrame:
    df_ret = None

    df = df_main.query("offstudy_reason!='completed_followup'")

    df6 =  df.query("primary_cohort_str.isin(@cohorts) and retained_months<=6 and assignment==@arm").offstudy_reason.value_counts(dropna=False).to_frame().reset_index()
    df6.columns = ["offstudy_reason", "freq"]
    df6 = df6.reset_index(drop=True).set_index("offstudy_reason")

    df_all = df.query("primary_cohort_str.isin(@cohorts) and assignment==@arm").offstudy_reason.value_counts(dropna=False).to_frame().reset_index()
    df_all.columns = ["offstudy_reason", "freq"]
    df_all = df_all.reset_index(drop=True).set_index("offstudy_reason")

    if attr == "6m":
        df_ret = df6.fillna(0)
    elif attr == "12m":
        df_ret = df_all.fillna(0)
    df_ret = df_ret.reset_index().rename(columns={'index':'offstudy_reason'})
    df_ret["offstudy_reason"] = df_ret["offstudy_reason"].apply(lambda x:x.lower())
    df_ret = df_ret.sort_values("offstudy_reason")
    return df_ret

def render_results(attr:str, visit_code:float, months:int, primary_cohorts:list[str], assignment:str):
    onstudy = len(get_onstudy_at(attr, primary_cohorts, assignment))
    df_offstudy_reasons = get_offstudy_reasons(attr, primary_cohorts, assignment)
    baseline = len(df_main.query("primary_cohort_str.isin(@primary_cohorts) and assignment==@assignment"))
    perc = onstudy/baseline
    print(f"{onstudy}/{baseline} ({perc * 100}) onstudy/baseline (%)")
    print(f"{visit_code} {months}m [{', '.join(primary_cohorts)}] assignment={assignment}\n", onstudy)
    print(df_offstudy_reasons)
    print(f"TOTAL: {df_offstudy_reasons.sum().freq}")



In [ ]:
visit_code = 1060.0
months = 6
primary_cohorts = ['DM_ALONE','HTN_DM','HTN_ALONE']
assignment = "a"
attr = "6m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1060.0
months = 6
primary_cohorts = ['DM_ALONE','HTN_DM','HTN_ALONE']
assignment = "b"
attr = "6m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1060.0
months = 6
primary_cohorts = ['HIV_ALONE']
assignment = "a"
attr = "6m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1060.0
months = 6
primary_cohorts = ['HIV_ALONE']
assignment = "b"
attr = "6m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1120.0
months = 12
primary_cohorts = ['DM_ALONE','HTN_DM','HTN_ALONE']
assignment = "a"
attr = "12m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1120.0
months = 12
primary_cohorts = ['DM_ALONE','HTN_DM','HTN_ALONE']
assignment = "b"
attr = "12m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1120.0
months = 12
primary_cohorts = ['HIV_ALONE']
assignment = "a"
attr = "12m"
render_results(attr, visit_code, months, primary_cohorts, assignment)

In [ ]:
visit_code = 1120.0
months = 12
primary_cohorts = ['HIV_ALONE']
assignment = "b"
attr = "12m"
render_results(attr, visit_code, months, primary_cohorts, assignment)